# Tutorial 2: Run Retrieval Evaluation on CoREB

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hq-bench/coreb/blob/main/notebooks/02_run_evaluation.ipynb)

In this notebook, you will learn how to:
1. Load CoREB data and convert it to evaluation format
2. Initialize an embedding model (HuggingFace or Gemini)
3. Run dense retrieval with `DenseRetrievalExactSearch`
4. Evaluate with graded-relevance metrics (nDCG, Recall, MAP, MRR)

**GPU recommended** — go to Runtime > Change runtime type > T4 GPU.

**Reference:** Xue et al., *Beyond Retrieval: A Multitask Benchmark and Model for Code Search*, 2025. [arXiv:2605.04615](https://arxiv.org/abs/2605.04615)

## 1. Installation

In [ ]:
!pip install -q "coreb[hf]"

## 2. Load Data from HuggingFace

We load the **v202603 (test)** split and convert it into the dict format expected by the evaluator.

In [ ]:
from datasets import load_dataset
from coreb_runner.benchmark import (
    convert_corpus_to_coir_format,
    convert_queries_to_coir_format,
    convert_qrels_to_coir_format,
)

SPLIT = "release_v2603"

# Load code corpus and T2C task as an example
code_corpus_hf = load_dataset("hq-bench/coreb", "code_corpus",       split=SPLIT)
t2c_queries_hf = load_dataset("hq-bench/coreb", "text2code_queries", split=SPLIT)
t2c_qrels_hf   = load_dataset("hq-bench/coreb", "text2code_qrels",   split=SPLIT)

# Convert to evaluation format
corpus  = convert_corpus_to_coir_format(list(code_corpus_hf))
queries = convert_queries_to_coir_format(list(t2c_queries_hf))
qrels   = convert_qrels_to_coir_format(list(t2c_qrels_hf))

print(f"Corpus:  {len(corpus):,} documents")
print(f"Queries: {len(queries):,}")
print(f"Qrels:   {len(qrels):,} query groups")

## 3. Initialize an Embedding Model

CoREB supports two model backends:
- **HuggingFace** (`model_type="huggingface"`): any Transformers-compatible encoder
- **Gemini** (`model_type="gemini"`): Google's embedding API (requires `GEMINI_API_KEY`)

Here we use a HuggingFace model as an example.

In [ ]:
from coreb_runner.benchmark import create_model_wrapper

# Choose a model — any HuggingFace encoder works.
# Smaller models are faster for demonstration:
MODEL_NAME = "jinaai/jina-embeddings-v3"

model = create_model_wrapper(MODEL_NAME, model_type="huggingface")
print(f"Model loaded: {MODEL_NAME}")

## 4. Run Dense Retrieval

`DenseRetrievalExactSearch` encodes all queries and corpus documents, then computes brute-force cosine similarity to find top-k results.

In [ ]:
from coreb_runner.benchmark import DenseRetrievalExactSearch, EvaluateRetrieval

K_VALUES = [1, 3, 5, 10]

# Build retriever and evaluator
retriever = DenseRetrievalExactSearch(model, batch_size=64)
evaluator = EvaluateRetrieval(retriever, k_values=K_VALUES)

# Run retrieval
results = evaluator.retrieve(corpus, queries)

print(f"Retrieved results for {len(results):,} queries")

# Peek at top-3 results for the first query
first_qid = next(iter(results))
top3 = sorted(results[first_qid].items(), key=lambda x: x[1], reverse=True)[:3]
print(f"\nTop-3 for query '{first_qid}':")
for doc_id, score in top3:
    print(f"  {doc_id}: {score:.4f}")

## 5. Evaluate with Graded Relevance

CoREB uses `relevance_level=2`: only true positives (rel>=2) count as relevant for binary metrics (Recall, MAP, Precision). Hard negatives (rel=1) penalize nDCG by occupying top ranks with zero gain.

In [ ]:
ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(
    qrels, results, K_VALUES
)

print("=" * 40)
print(f"  Model: {MODEL_NAME}")
print(f"  Task:  Text-to-Code (T2C)")
print("=" * 40)
for k in K_VALUES:
    print(f"  nDCG@{k:<3d}  {ndcg[f'NDCG@{k}']:.4f}")
print("-" * 40)
for k in K_VALUES:
    print(f"  Recall@{k:<3d}{recall[f'Recall@{k}']:.4f}")
print("-" * 40)
for k in K_VALUES:
    print(f"  MAP@{k:<3d}   {_map[f'MAP@{k}']:.4f}")
print("-" * 40)
for k in K_VALUES:
    print(f"  P@{k:<3d}     {precision[f'P@{k}']:.4f}")

## 6. Custom Metrics: MRR and Top-k Accuracy

In [ ]:
mrr = EvaluateRetrieval.evaluate_custom(qrels, results, K_VALUES, metric="mrr")
acc = EvaluateRetrieval.evaluate_custom(qrels, results, K_VALUES, metric="accuracy")

print("MRR (only rel>=2 count as hits):")
for k in K_VALUES:
    print(f"  MRR@{k}: {mrr[f'MRR@{k}']:.4f}")

print("\nTop-k Accuracy:")
for k in K_VALUES:
    print(f"  Accuracy@{k}: {acc[f'Accuracy@{k}']:.4f}")

## 7. Evaluate All Three Tasks

Use `CoREBEvaluation` to run retrieval and evaluation across multiple tasks in one go.

In [ ]:
from coreb_runner.benchmark import CoREBEvaluation

# Load remaining tasks
text_corpus_hf = load_dataset("hq-bench/coreb", "text_corpus", split=SPLIT)

c2c_queries_hf = load_dataset("hq-bench/coreb", "code2code_queries", split=SPLIT)
c2c_qrels_hf   = load_dataset("hq-bench/coreb", "code2code_qrels",   split=SPLIT)

c2t_queries_hf = load_dataset("hq-bench/coreb", "code2text_queries", split=SPLIT)
c2t_qrels_hf   = load_dataset("hq-bench/coreb", "code2text_qrels",   split=SPLIT)

# Convert all to eval format
text_corpus_eval = convert_corpus_to_coir_format(list(text_corpus_hf))

c2c_queries_eval = convert_queries_to_coir_format(list(c2c_queries_hf))
c2c_qrels_eval   = convert_qrels_to_coir_format(list(c2c_qrels_hf))

c2t_queries_eval = convert_queries_to_coir_format(list(c2t_queries_hf))
c2t_qrels_eval   = convert_qrels_to_coir_format(list(c2t_qrels_hf))

# Define tasks: {name: (corpus, queries, qrels)}
tasks = {
    "text2code": (corpus, queries, qrels),                            # T2C: query=text, corpus=code
    "code2code": (corpus, c2c_queries_eval, c2c_qrels_eval),         # C2C: query=code, corpus=code
    "code2text": (text_corpus_eval, c2t_queries_eval, c2t_qrels_eval), # C2T: query=code, corpus=text
}

# Run all tasks
coreb_eval = CoREBEvaluation(tasks, batch_size=64)
all_results = coreb_eval.run(model, output_folder="./coreb_results")

# Print summary
print("\n" + "=" * 50)
print(f"{'Task':15s} {'nDCG@10':>10s}")
print("=" * 50)
for task_name, metrics in all_results.items():
    ndcg10 = metrics["NDCG"]["NDCG@10"]
    print(f"{task_name:15s} {ndcg10:>10.4f}")

## 8. Using the Gemini Embedding API (Optional)

To use Google's Gemini embedding models instead, set your API key and switch the model type.

In [ ]:
# Uncomment to use Gemini:

# import os
# os.environ["GEMINI_API_KEY"] = "your-api-key-here"
#
# gemini_model = create_model_wrapper(
#     "gemini-embedding-2-preview",
#     model_type="gemini",
#     batch_size=100,
# )
#
# retriever = DenseRetrievalExactSearch(gemini_model, batch_size=100)
# evaluator = EvaluateRetrieval(retriever, k_values=[1, 3, 5, 10])
# results = evaluator.retrieve(corpus, queries)
# ndcg, _map, recall, precision = evaluator.evaluate(qrels, results, [1, 3, 5, 10])
# print(f"Gemini nDCG@10: {ndcg['NDCG@10']:.4f}")

## Citation

If you use CoREB in your research, please cite:

```bibtex
@article{xue2025coreb,
  title   = {Beyond Retrieval: A Multitask Benchmark and Model for Code Search},
  author  = {Xue, Siqiao and Liao, Zihan and Qin, Jin and Zhang, Ziyin and Mu, Yixiang and Zhou, Fan and Yu, Hang},
  journal = {arXiv preprint arXiv:2605.04615},
  year    = {2025},
  url     = {https://arxiv.org/abs/2605.04615}
}
```